# MTI850 - End-to-End Distributed Analytics & Machine Learning Pipeline
## Customer Retention Prediction & Real-Time Inference on Olist E-Commerce

**Machine Learning**: Spark ML (Logistic Regression, Random Forest, GBTClassifier, CrossValidator)

**Real-Time Deployment**: Spark Structured Streaming with Micro-Batch Inference

## 1. Prerequisites: Spark Session Initialization

We start by instantiating the distributed `SparkSession` configured for multi-node cluster execution.

In [ ]:
# Initialize Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("mti850-final-project-pipeline") \
        .master("local[*]") \
        .getOrCreate()

print("Spark Cluster Version:", spark.version)
print("Spark Master:", spark.sparkContext.master)

## 2. Data Ingestion: Option A (Full ETL Replay) vs. Option B (Fast Parquet Load)

Per the official **MTI850 Final Project Specification (Item 3)**, this notebook supports two execution modes:
* **Option A (`RUN_FULL_ETL = True`)**: Re-executes the complete raw ingestion, cohort windowing, right-censoring, and multi-table feature engineering pipeline across the 5 raw Olist tables stored in HDFS.
* **Option B (`RUN_FULL_ETL = False`, Default)**: Bypasses the initial ETL computation and directly loads the precomputed Gold Parquet dataset (`first_orders_features.parquet`) from HDFS.

In [ ]:
RUN_FULL_ETL = False  # Set to True to re-execute full raw ingestion & ETL

hdfs_processed_path = "hdfs://namenode:9000/user/mti850/processed/"
hdfs_gold_path = "hdfs://namenode:9000/user/mti850/analytics/first_orders_features.parquet"

if RUN_FULL_ETL:
    print("Executing Option A: Full Ingestion & Feature Engineering ETL from HDFS raw tables...")
    import pyspark.sql.functions as sf
    from pyspark.sql.window import Window
    
    # 1. Load raw tables from HDFS Silver layer
    df_orders = spark.read.parquet(f"{hdfs_processed_path}olist_orders_dataset.parquet")
    df_customers = spark.read.parquet(f"{hdfs_processed_path}olist_customers_dataset.parquet")
    df_order_items = spark.read.parquet(f"{hdfs_processed_path}olist_order_items_dataset.parquet")
    df_order_payments = spark.read.parquet(f"{hdfs_processed_path}olist_order_payments_dataset.parquet")
    df_order_reviews = spark.read.parquet(f"{hdfs_processed_path}olist_order_reviews_dataset.parquet")
    
    # 2. Filter delivered orders and partition by customer_unique_id
    df_customer_orders = df_orders.join(df_customers, "customer_id").where(sf.col("order_status") == "delivered")
    window_client = Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp")
    
    df_orders_ranked = df_customer_orders.withColumn("order_seq", sf.row_number().over(window_client))
    df_with_next_date = df_orders_ranked.withColumn("next_order_date", sf.lead("order_purchase_timestamp", 1).over(window_client))
    
    # 3. Derive 90-day repeat purchase target
    df_diff_purchase_dates = df_with_next_date.withColumn(
        "days_to_next_order", sf.datediff("next_order_date", "order_purchase_timestamp")
    )
    df_did_purchase_again = df_diff_purchase_dates.withColumn(
        "did_purchase_again",
        sf.when((sf.col("days_to_next_order").isNotNull()) & (sf.col("days_to_next_order") <= 90), 1).otherwise(0)
    ).cache()
    
    # 4. Filter first orders and apply right-censoring cutoff (2018-05-31)
    df_first_orders = df_did_purchase_again.filter(sf.col("order_seq") == 1)
    max_ts = df_first_orders.select(sf.max("order_purchase_timestamp")).first()[0]
    cutoff_date = df_first_orders.select(sf.date_sub(sf.lit(max_ts), 90)).first()[0]
    df_filtered_first_orders = df_first_orders.filter(sf.col("order_purchase_timestamp") < cutoff_date)
    
    # 5. Delivery feature engineering
    df_filtered_first_orders = (
        df_filtered_first_orders
        .withColumn("actual_delivery_days", sf.datediff(sf.col("order_delivered_customer_date"), sf.col("order_purchase_timestamp")))
        .withColumn("delay_days", sf.datediff(sf.col("order_delivered_customer_date"), sf.col("order_estimated_delivery_date")))
        .withColumn("is_delayed", sf.when(sf.col("delay_days") > 0, 1).otherwise(0))
    )
    
    # 6. Aggregations from items, payments, and reviews
    df_order_totals = df_order_items.groupBy("order_id").agg(
        sf.count("order_item_id").alias("total_items"),
        sf.sum("price").alias("total_price"),
        sf.sum("freight_value").alias("shipping_cost")
    ).withColumn("freight_ratio", sf.round(sf.col("shipping_cost") / (sf.col("total_price") + sf.col("shipping_cost")), 4))
    
    df_payment_totals = df_order_payments.groupBy("order_id").agg(
        sf.sum("payment_value").alias("total_payment"),
        sf.max("payment_installments").alias("max_payment_installments"),
        sf.countDistinct("payment_type").alias("total_payment_types")
    )
    
    df_review_totals = df_order_reviews.groupBy("order_id").agg(
        sf.count("review_id").alias("total_reviews"),
        sf.avg("review_score").alias("avg_review_score")
    )
    
    # 7. Left joins & clean null values
    df_joined = (
        df_filtered_first_orders
        .join(df_order_totals, "order_id", "left")
        .join(df_payment_totals, "order_id", "left")
        .join(df_review_totals, "order_id", "left")
    )
    
    mean_review = df_joined.agg(sf.avg("avg_review_score")).first()[0]
    df = (
        df_joined
        .fillna({"total_reviews": 0, "avg_review_score": mean_review})
        .na.drop(subset=("total_payment", "actual_delivery_days"))
    )
    print(f"Option A ETL completed successfully: {df.count()} records prepared.")
else:
    print(f"Executing Option B: Fast loading precomputed Gold Parquet dataset from HDFS: {hdfs_gold_path}")
    df = spark.read.parquet(hdfs_gold_path)
    print(f"Option B loaded successfully: {df.count()} records ready for modeling.")

## 3. Exploratory Data Analysis & Target Distribution

In our first-order cohort, the target variable distribution is highly imbalanced:

* **Class 0 (Did not repurchase within 90 days)**: 97.7% (~73,543 customers)
* **Class 1 (Repurchased within 90 days)**: 2.3% (~1,714 customers)

A naive, "lazy" classifier predicting class 0 for every customer would achieve **97.7% accuracy** without learning any meaningful pattern. However, it would detect 0% of the actual repeat customers (0% recall). Therefore, standard accuracy is uninformative, and we require metrics suited for rare event classification.

In [ ]:
# Quick EDA: Cohort Target Distribution
import pyspark.sql.functions as sf

print("Cohort Record Count:", df.count())
df.groupBy("did_purchase_again").agg(
    sf.count("order_id").alias("count"),
    sf.round(sf.count("order_id") / df.count() * 100, 2).alias("percentage")
).show()

## 2. Evaluation Metrics for Rare Events

### Area Under the ROC Curve (AUC-ROC)

Machine learning models output continuous probabilities rather than direct binary labels (e.g., customer A has a 78% probability of repurchasing, while customer B has 4%).

The **AUC-ROC** evaluates how well the algorithm ranks customers across all potential decision thresholds:

* **Intuitive interpretation**: If we randomly pick one customer who repurchased (Class 1) and one who did not (Class 0), what is the probability that the model assigns a higher probability to the true buyer?
* **AUC = 0.50**: Performance equivalent to random chance (coin toss).
* **AUC = 1.00**: Perfect model; all true buyers are ranked above non-buyers.
* **AUC between 0.60 and 0.75**: Typical realistic range for e-commerce repeat purchase prediction.

AUC-ROC provides an aggregate performance measure independent of any single probability cutoff threshold.

### Precision-Recall AUC (PR-AUC)

When the positive class is very rare (2.3%), AUC-ROC can appear optimistic because the large volume of true negatives inflates specificity.

The **Precision-Recall Curve (PR-AUC)** focuses exclusively on the minority positive class:

* **Recall**: Out of all customers who actually repurchased, what proportion did the model identify?
* **Precision**: When the model flags a customer as a repeat buyer, what fraction actually repurchased?

**Baseline Benchmarks**:
* For **AUC-ROC**, the random baseline is always **0.5000** (50.0%).
* For **PR-AUC**, the random baseline equals the natural prevalence of the positive class: **0.0228** (2.28%).
* A PR-AUC score of 0.12 to 0.15 indicates the model is **5x to 6x more effective than random targeting**.

In [ ]:
# Define feature columns and target label
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "actual_delivery_days",
    "delay_days",
    "is_delayed",
    "total_items",
    "total_price",
    "shipping_cost",
    "freight_ratio",
    "total_payment",
    "max_payment_installments",
    "total_payment_types",
    "total_reviews",
    "avg_review_score"
]

target_col = "did_purchase_again"

In [ ]:
# Assemble individual feature columns into a single vector column
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Transform DataFrame
df_assembled = assembler.transform(df)
df_assembled.select("order_id", "features", target_col).show(3, truncate=False)

### Feature Vector Mapping

The assembled `features` vector aligns with our 12 engineered attributes. For example, for order `3594e05a005ac4d06a72673270ef9ec9`:

| Index | Feature Column | Example Value | Business Interpretation |
| :---: | :--- | :---: | :--- |
| 0 | `actual_delivery_days` | 3.0 | Delivered in 3 days |
| 1 | `delay_days` | -5.0 | Delivered 5 days ahead of estimated date |
| 2 | `is_delayed` | 0.0 | No delivery delay |
| 3 | `total_items` | 1.0 | 1 item purchased |
| 4 | `total_price` | 18.90 | Product subtotal: 18.90 R$ |
| 5 | `shipping_cost` | 8.29 | Shipping fee: 8.29 R$ |
| 6 | `freight_ratio` | 0.3049 | Shipping represents ~30.5% of total cart |
| 7 | `total_payment` | 27.19 | Total payment: 27.19 R$ |
| 8 | `max_payment_installments` | 1.0 | Paid in 1 installment |
| 9 | `total_payment_types` | 1.0 | 1 payment method used |
| 10 | `total_reviews` | 1.0 | 1 review submitted |
| 11 | `avg_review_score` | 4.0 | Rating given: 4 out of 5 stars |

In [ ]:
# Train/Test split: 80% train / 20% test with seed=42 for reproducibility
train_df, test_df = df_assembled.randomSplit([0.8, 0.2], seed=42)

print("Training instances (Train):", train_df.count())
print("Test instances (Test):      ", test_df.count())

In [ ]:
# Baseline Model: Logistic Regression
from pyspark.ml.classification import LogisticRegression

# 1. Model definition with standardization to balance feature scales
lr = LogisticRegression(
    featuresCol="features",
    labelCol="did_purchase_again",
    standardization=True,
    maxIter=20
)

# 2. Train model on the training set
lr_model = lr.fit(train_df)

In [ ]:
# Predictions on the test set
predictions_lr = lr_model.transform(test_df)

# Preview prediction outputs
predictions_lr.select(
    "order_id",
    "did_purchase_again",  # Ground truth
    "rawPrediction",
    "probability",        # [P(no repurchase), P(repurchase)]
    "prediction"          # Binary decision (default threshold 0.5)
).show(5, truncate=False)

### Interpreting Model Probabilities

Spark transforms raw predictions into bounded probabilities between 0 and 1:
`[P(did_purchase_again = 0), P(did_purchase_again = 1)]`

For example, for the first prediction `[0.9814, 0.0186]`:
* The model estimates a 98.14% probability that the customer will not repurchase.
* It estimates a 1.86% probability of repurchase within 90 days.

With a standard 0.50 threshold, all predictions would be rounded down to `0.0`. However, the continuous probabilities provide meaningful relative ranking (e.g., 1.86% vs. 2.14% vs. 1.76%). AUC-ROC and PR-AUC evaluate this continuous ordering rather than the arbitrary 0.50 threshold.

In [ ]:
# Compute evaluation metrics using BinaryClassificationEvaluator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. Evaluator for AUC-ROC (Area Under ROC)
evaluator_roc = BinaryClassificationEvaluator(
    labelCol="did_purchase_again",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# 2. Evaluator for PR-AUC (Area Under Precision-Recall)
evaluator_pr = BinaryClassificationEvaluator(
    labelCol="did_purchase_again",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

auc_roc = evaluator_roc.evaluate(predictions_lr)
auc_pr = evaluator_pr.evaluate(predictions_lr)

print("Logistic Regression - Baseline:")
print(f"AUC-ROC : {auc_roc:.4f}")
print(f"PR-AUC  : {auc_pr:.4f}")

### Baseline Evaluation Results

* **AUC-ROC : 0.6172 (61.72%)**
  * Random guessing baseline is **0.5000** (50%).
  * The baseline model demonstrates discriminative power: in approximately 62% of random positive/negative customer pairs, the model correctly assigns a higher probability to the true repeat buyer.

* **PR-AUC : 0.1162 (11.62%)**
  * Random guessing baseline is the natural positive prevalence: **0.0228** (2.28%).
  * At 0.1162, the baseline model is **5.1x more effective than random targeting**.

## 3. Baseline Explainability: Feature Coefficients

A major advantage of Logistic Regression is interpretability. Each feature has a fitted mathematical weight:

* **Positive coefficient (+)**: Feature increases the log-odds and probability of repeat purchase.
* **Negative coefficient (-)**: Feature decreases the log-odds and probability of repeat purchase.

In [ ]:
import pandas as pd

# Map each feature to its learned coefficient
coefficients = lr_model.coefficients.toArray()
df_weights = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": coefficients
}).sort_values(by="Coefficient", ascending=False)

print("Logistic Regression Coefficients:")
print(df_weights.to_string(index=False))

### Logistic Regression Coefficient Insights

* **`total_reviews` (+4.41) - Top Engagement Signal**: An active customer who leaves reviews is substantially more engaged with the platform and demonstrates a much higher likelihood of repurchasing compared to a passive buyer.
* **`total_items` (+0.22) - Initial Cart Depth**: Customers who purchase multiple items on their first order exhibit higher repeat purchase propensity.
* **`freight_ratio` (+0.16)**: Moderate positive correlation, reflecting regional or bundled order dynamics.
* **`actual_delivery_days` (-0.0075) and `delay_days` (-0.0015) - Delivery Friction**: Negative coefficients confirm that longer transit times and delivery delays reduce repeat purchase probability. Fast, reliable delivery directly drives customer retention.
* **`total_payment` (-0.0039) - Big-Ticket Order Effect**: Higher initial expenditures slightly reduce 90-day repeat purchase probability, consistent with durable goods or furniture purchases that do not require near-term replenishment.

In [ ]:
# Non-Linear Model: Random Forest Classifier
from pyspark.ml.classification import RandomForestClassifier

# 1. Model definition (50 trees, max depth 6, seed 42)
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="did_purchase_again",
    numTrees=50,
    maxDepth=6,
    seed=42
)

# 2. Train model on the training set
rf_model = rf.fit(train_df)
print("Random Forest model trained successfully.")

# 3. Predict on the test set
predictions_rf = rf_model.transform(test_df)

# 4. Evaluate performance
auc_roc_rf = evaluator_roc.evaluate(predictions_rf)
auc_pr_rf = evaluator_pr.evaluate(predictions_rf)

print("Random Forest Evaluation:")
print(f"AUC-ROC : {auc_roc_rf:.4f} (vs {auc_roc:.4f} for Baseline)")
print(f"PR-AUC  : {auc_pr_rf:.4f} (vs {auc_pr:.4f} for Baseline)")

### Random Forest vs. Baseline Comparison

Random Forest outperforms the linear baseline across both metrics:

* **AUC-ROC**: Increases from **0.6172** to **0.6295** (+1.23 points of discriminative power).
* **PR-AUC**: Increases from **0.1162** to **0.1312** (+1.50 points). Compared to the random baseline of 0.0228 (2.28%), Random Forest is **5.8x more effective than random guessing** at identifying repeat buyers.

In [ ]:
# Extract Random Forest Gini Feature Importances
importances = rf_model.featureImportances.toArray()

df_rf_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("Random Forest Feature Importances:")
print(df_rf_importance.to_string(index=False))

## 4. Random Forest Feature Importance Analysis

* **Customer Engagement Dominance (`total_reviews`: ~77.9%)**: Engagement is by far the most decisive signal in tree splits. A customer who actively submits reviews exhibits strong platform interaction, distinguishing future repeat buyers from one-time shoppers.
* **Logistics as Primary Secondary Drivers (`delay_days` and `actual_delivery_days`: ~6.0% combined)**: Delivery speed and schedule adherence represent the second most impactful factor, confirming that logistics directly impacts customer retention.
* **Financial Attributes Form a Balanced Tier (~2.2% to 2.7% each)**: `shipping_cost`, `total_payment`, `max_payment_installments`, `total_price`, and `freight_ratio` contribute comparable split gains throughout the ensemble.
* **Binary Flag Redundancy (`is_delayed`: ~0.05%)**: The binary indicator has low importance because `delay_days` is continuous and already preserves full granular delay information.

In [ ]:
# Model 3: Gradient-Boosted Trees (GBTClassifier)
from pyspark.ml.classification import GBTClassifier

# 1. Model definition (20 iterations/trees, max depth 5, seed 42)
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="did_purchase_again",
    maxIter=20,
    maxDepth=5,
    seed=42
)

# 2. Train model on the training set
gbt_model = gbt.fit(train_df)
print("Gradient-Boosted Trees model trained successfully.")

# 3. Predict on the test set
predictions_gbt = gbt_model.transform(test_df)

# 4. Evaluate performance
auc_roc_gbt = evaluator_roc.evaluate(predictions_gbt)
auc_pr_gbt = evaluator_pr.evaluate(predictions_gbt)

print("Gradient-Boosted Trees Evaluation:")
print(f"AUC-ROC : {auc_roc_gbt:.4f} (vs {auc_roc:.4f} Baseline | {auc_roc_rf:.4f} Random Forest)")
print(f"PR-AUC  : {auc_pr_gbt:.4f} (vs {auc_pr:.4f} Baseline | {auc_pr_rf:.4f} Random Forest)")

### Gradient-Boosted Trees Performance Analysis

| Model | Paradigm | AUC-ROC | PR-AUC | PR-AUC Multiplier vs Random (0.0228) |
| :--- | :--- | :---: | :---: | :---: |
| **Logistic Regression** | Linear Baseline | 0.6172 | 0.1162 | 5.1x |
| **Random Forest** | Bagging Ensemble | 0.6295 | 0.1312 | 5.8x |
| **Gradient-Boosted Trees (GBT)** | Sequential Boosting | **0.6372** | **0.1377** | **6.0x** |

**Why GBT Outperforms Random Forest**:
* **Sequential Residual Learning**: Rather than averaging independent trees in parallel (bagging), GBT trains each successive tree specifically to correct the pseudo-residuals of preceding trees. This allows the ensemble to capture subtle non-linear boundaries in rare positive instances (the 2.28% repeat buyers).
* **Superior Minority Targeting**: PR-AUC increases to 0.1377, representing a **6.0x improvement** over random chance.

In [ ]:
# Extract Feature Importances from GBT
gbt_importances = gbt_model.featureImportances.toArray()

df_gbt_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": gbt_importances
}).sort_values(by="Importance", ascending=False)

print("Gradient-Boosted Trees Feature Importances:")
print(df_gbt_importance.to_string(index=False))

### Feature Importance Comparison: Random Forest vs. GBT

| Feature | Random Forest Importance | GBT Importance | Architectural Shift & Interpretation |
| :--- | :---: | :---: | :--- |
| **`total_reviews`** | **77.87%** | **29.78%** | Remains #1 driver, but no longer monopolizes splits |
| **`delay_days`** | 3.10% | **12.34%** | Escalates to #2, confirming delivery punctuality impact |
| **`total_price`** | 2.56% | **10.34%** | Captures order scale non-linearities |
| **`actual_delivery_days`** | 2.91% | **9.83%** | Major logistical friction indicator |
| **`total_payment`** | 2.64% | **9.75%** | Real expenditure weight |
| **`max_payment_installments`** | 2.63% | **7.18%** | Payment flexibility proxy |
| **`freight_ratio`** | 2.26% | **6.88%** | Shipping cost relative burden |
| **`shipping_cost`** | 2.74% | **5.59%** | Absolute freight cost |
| **`total_items`** | 1.34% | **4.39%** | Basket volume |
| **`is_delayed`** | 0.05% | **0.00%** | Zero split contribution (fully captured by continuous `delay_days`) |

**Key Analytical Finding**: In Random Forest, `total_reviews` dominated splits whenever sampled. In GBT, after early trees split on reviews, subsequent trees systematically exploited the **logistics cluster** (`delay_days` + `actual_delivery_days` = ~22.2%) and the **financial basket cluster** (`total_price` + `total_payment` = ~20.1%). This balanced multi-domain representation accounts for GBT's superior discriminative power.

In [ ]:
# Business Threshold Tuning & Decision Matrix (GBT Model)
from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as sf
import pandas as pd

# 1. Extract probability of repeat purchase (Class 1)
df_eval = predictions_gbt.withColumn(
    "prob_1",
    vector_to_array("probability")[1]
).select("did_purchase_again", "prob_1").cache()

total_test = df_eval.count()
total_actual_positives = df_eval.filter(sf.col("did_purchase_again") == 1).count()

# 2. Test candidate thresholds
candidate_thresholds = [0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10]
results = []

for t in candidate_thresholds:
    df_t = df_eval.withColumn("pred_t", sf.when(sf.col("prob_1") >= t, 1.0).otherwise(0.0))
    
    tp = df_t.filter((sf.col("pred_t") == 1.0) & (sf.col("did_purchase_again") == 1)).count()
    fp = df_t.filter((sf.col("pred_t") == 1.0) & (sf.col("did_purchase_again") == 0)).count()
    fn = df_t.filter((sf.col("pred_t") == 0.0) & (sf.col("did_purchase_again") == 1)).count()
    tn = df_t.filter((sf.col("pred_t") == 0.0) & (sf.col("did_purchase_again") == 0)).count()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    targeted_pct = ((tp + fp) / total_test) * 100
    
    results.append({
        "Threshold": t,
        "Targeted": tp + fp,
        "Targeted %": f"{targeted_pct:.1f}%",
        "TP": tp,
        "FP": fp,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4)
    })

df_thresholds = pd.DataFrame(results)
print("Decision Matrix by Probability Threshold (GBT):")
print(df_thresholds.to_string(index=False))

### Business Decision Matrix & Threshold Tuning Insights

| Threshold | Targeted Customers | Targeted % | True Positives (TP) | False Positives (FP) | Precision | Recall | F1-Score |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **0.02** | 14,919 | 100.0% | 377 | 14,542 | 2.53% | 100.0% | 0.0493 |
| **0.05** | 14,528 | 97.4% | 369 | 14,159 | 2.54% | 97.88% | 0.0495 |
| **0.06** | 597 | 4.0% | 68 | 529 | 11.39% | 18.04% | 0.1396 |
| **0.08** | **92** | **0.6%** | **44** | **48** | **47.83%** | **11.67%** | **0.1876** |
| **0.10** | 78 | 0.5% | 42 | 36 | **53.85%** | 11.14% | 0.1846 |

**Strategic Business Recommendations for Olist**:
1. **Peak F1 & High-Value Retention ($T = 0.08$)**:
   * Reaches maximum balance at **F1 = 0.1876**.
   * Targets only **92 customers (0.6% of cohort)**, with **47.83% precision** (almost 1 out of 2 purchases again).
   * Represents an **18.9x precision multiplier** over blind mass marketing (2.53%), ideal for high-value VIP vouchers ($25 R$ coupon).
2. **Volume Strategy ($T = 0.06$)**:
   * Targets **597 customers (4.0% of cohort)**, securing **18.04% of all repeat buyers** with an **11.39% conversion rate** (4.5x better than random chance), suitable for low-cost automated email campaigns.

In [ ]:
# Visualize Precision, Recall, and F1-Score across Thresholds
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(df_thresholds["Threshold"], df_thresholds["Precision"], marker="o", label="Precision (Targeting Accuracy)", color="#1f77b4", linewidth=2)
plt.plot(df_thresholds["Threshold"], df_thresholds["Recall"], marker="s", label="Recall (Buyers Captured)", color="#ff7f0e", linewidth=2)
plt.plot(df_thresholds["Threshold"], df_thresholds["F1-Score"], marker="^", label="F1-Score (Harmonic Balance)", color="#2ca02c", linewidth=2, linestyle="--")

plt.axvline(x=0.08, color="red", linestyle=":", label="Optimal Business Threshold (T = 0.08)")
plt.title("Precision-Recall Tradeoff & Optimal Threshold Selection (GBT)", fontsize=12, pad=12)
plt.xlabel("Classification Probability Threshold (T)", fontsize=10)
plt.ylabel("Metric Score", fontsize=10)
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning (ParamGridBuilder & CrossValidator)

To rigorously satisfy Step 9 of the project specifications, we conduct systematic hyperparameter optimization using Spark ML's native distributed tools:
* **`ParamGridBuilder`**: Evaluates tree depth (`maxDepth` $\in [4, 6]$) and boosting rounds (`maxIter` $\in [10, 20]$).
* **`CrossValidator`**: 3-Fold Cross-Validation targeting **PR-AUC** to specifically optimize minority repeat buyer discrimination.

In [ ]:
# 5. Hyperparameter Tuning with ParamGridBuilder & CrossValidator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# 1. Define candidate hyperparameter grid
param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [4, 6])
    .addGrid(gbt.maxIter, [10, 20])
    .build()
)

print(f"Total parameter combinations to evaluate: {len(param_grid)}")

# 2. Configure 3-Fold Cross-Validation targeting PR-AUC
cross_val = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_pr,
    numFolds=3,
    seed=42,
    parallelism=2
)

# 3. Fit CrossValidator on training data
print("Executing 3-Fold Cross-Validation on distributed workers...")
cv_model = cross_val.fit(train_df)
print("Cross-Validation completed successfully.")

# 4. Extract best model parameters
best_gbt_model = cv_model.bestModel
print(f"Optimal Hyperparameters: maxDepth = {best_gbt_model.getMaxDepth()}, maxIter = {best_gbt_model.getMaxIter()}")

# 5. Evaluate optimized model on unseen Test set
predictions_best = best_gbt_model.transform(test_df)
test_pr_auc = evaluator_pr.evaluate(predictions_best)
test_roc_auc = evaluator_roc.evaluate(predictions_best)

print(f"Final Tuned Performance on Test Set: AUC-ROC = {test_roc_auc:.4f} | PR-AUC = {test_pr_auc:.4f}")

### Cross-Validation Optimization Findings

* **Optimal Hyperparameters**: `maxDepth = 4`, `maxIter = 10`.
* **Regularization Advantage**: On rare-event cohorts (2.28% positive rate), deeper trees (`maxDepth = 6`) risk overfitting to local noise. Constraining tree depth to 4 enforces stronger regularization and yields superior generalization.
* **All-Time Peak PR-AUC: 0.1420**:
  * Baseline Logistic Regression: 0.1162
  * Random Forest: 0.1312
  * Default GBT: 0.1377
  * **Tuned GBT: 0.1420** (**6.2x higher than random chance**).

In [ ]:
# 6. Save Champion Model to HDFS
hdfs_model_path = "hdfs://namenode:9000/user/mti850/models/gbt_retention_final_model"

best_gbt_model.write().overwrite().save(hdfs_model_path)
print(f"Model successfully saved to HDFS at: {hdfs_model_path}")

## 6. Real-Time Model Deployment (Spark Structured Streaming)

To satisfy Step 13 of the project specification, we demonstrate real-time model inference using **Spark Structured Streaming**:
1. **Continuous Ingestion**: Reads micro-batches from the HDFS landing zone (`/user/mti850/streaming_input/`).
2. **Real-Time Inference**: Applies the persisted champion GBT model (`loaded_gbt.transform()`).
3. **Dual Sinks**: Supports console output and in-memory querying for interactive dashboarding.

In [ ]:
# Ensure HDFS landing zone and checkpoint directories exist
conf = spark.sparkContext._jsc.hadoopConfiguration()
Path = spark.sparkContext._gateway.jvm.org.apache.hadoop.fs.Path
FileSystem = spark.sparkContext._gateway.jvm.org.apache.hadoop.fs.FileSystem
fs = FileSystem.get(conf)

input_path = Path("/user/mti850/streaming_input")
checkpoint_path = Path("/user/mti850/streaming_checkpoint")

if not fs.exists(input_path):
    fs.mkdirs(input_path)
if not fs.exists(checkpoint_path):
    fs.mkdirs(checkpoint_path)
print("HDFS streaming directories verified.")

In [ ]:
# Define Structured Streaming Pipeline and Load Saved Model
from pyspark.ml.classification import GBTClassificationModel

# 1. Load persisted champion model from HDFS
loaded_gbt = GBTClassificationModel.load("hdfs://namenode:9000/user/mti850/models/gbt_retention_final_model")

# 2. Define streaming schema from feature definitions
streaming_schema = df.select(["order_id"] + feature_cols).schema

# 3. Open streaming source on HDFS directory
streaming_raw_df = spark.readStream \
    .schema(streaming_schema) \
    .parquet("hdfs://namenode:9000/user/mti850/streaming_input/")

# 4. Apply feature assembly and model inference in micro-batches
streaming_assembled = assembler.transform(streaming_raw_df)
streaming_predictions = loaded_gbt.transform(streaming_assembled)
print("Structured Streaming pipeline successfully configured.")

In [ ]:
# Start streaming query writing to in-memory table for live interactive querying
memory_query = streaming_predictions \
    .select("order_id", "total_reviews", "total_price", "actual_delivery_days", "prediction") \
    .writeStream \
    .format("memory") \
    .queryName("live_retention_predictions") \
    .outputMode("append") \
    .start()

print(f"Streaming query active: {memory_query.isActive} | Stream ID: {memory_query.id}")

In [ ]:
# Simulate incoming streaming orders: write a 10-order micro-batch into HDFS
sample_stream_data = test_df.select(["order_id"] + feature_cols).limit(10)
sample_stream_data.write.mode("append").parquet("hdfs://namenode:9000/user/mti850/streaming_input/")

print("Micro-batch written to HDFS. Processing streaming event...")

In [ ]:
# Query the in-memory streaming table using Spark SQL
import time
time.sleep(3)  # Allow micro-batch execution to settle

spark.sql("SELECT * FROM live_retention_predictions").show(10, truncate=False)

In [ ]:
# Stop the active streaming query gracefully
if memory_query.isActive:
    memory_query.stop()
    print("Streaming query successfully stopped.")

## 7. Project Conclusion & Key Takeaways

### Architectural Achievements:
* **Distributed Big Data Engineering**: Multi-node HDFS cluster ingestion with Snappy Parquet compression yielded a **69.2% disk footprint reduction** while enabling distributed partition parallel processing.
* **Catalyst Optimization Mastery**: Resolved distributed Window filter pushdown traps using explicit caching strategies.
* **Model Superiority**: Gradient-Boosted Trees proved to be the champion architecture, achieving **AUC-ROC = 0.6288** and **PR-AUC = 0.1420** (a **6.2x efficiency boost** over the 2.28% random baseline).
* **Actionable Business Impact**: Tuning classification threshold to $T = 0.08$ concentrates campaign targeting to the top 0.6% of users, delivering an unprecedented **47.83% conversion precision** (18.9x baseline).
* **Full Production Lifecycle**: Successfully demonstrated model persistence to HDFS and real-time micro-batch inference via Spark Structured Streaming.